---
## Paso 2: Preprocesamiento del Dataset

Objetivos de este paso:
- Cargar y redimensionar imágenes a 64×64 px
- Normalizar píxeles al rango [0, 1]
- Aplicar aumentación de datos al conjunto de entrenamiento
- Dividir en entrenamiento (70%), validación (15%) y prueba (15%)
- Construir un pipeline eficiente con `tf.data` para alimentar la GPU


### 2.1 Importaciones del paso

In [ ]:
import os
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

# Silenciar logs de TF (solo errores)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"JAX backend : {jax.default_backend()}")


### 2.2 Recolección de rutas y etiquetas

Recorremos cada carpeta del dataset y construimos dos listas paralelas:
`rutas[]` con la ruta completa de cada imagen y `etiquetas[]` con su índice numérico de clase.


In [ ]:
rutas     = []
etiquetas = []

for clase in CLASES:
    ruta_clase = os.path.join(DATA_ROOT, clase)
    for archivo in os.listdir(ruta_clase):
        if archivo.lower().endswith((".jpg", ".jpeg", ".png")):
            rutas.append(os.path.join(ruta_clase, archivo))
            etiquetas.append(CLASE_A_IDX[clase])

rutas     = np.array(rutas)
etiquetas = np.array(etiquetas, dtype=np.int32)

print(f"Total de imágenes recolectadas : {len(rutas)}")
print(f"Total de etiquetas             : {len(etiquetas)}")
print()
print("Distribución por clase:")
conteo = Counter(etiquetas)
for idx in sorted(conteo):
    print(f"  {idx:2d} | {IDX_A_CLASE[idx]:<30s} : {conteo[idx]:4d}")


### 2.3 División del dataset

| Conjunto | Porcentaje | Aprox. imágenes |
|----------|-----------|-----------------|
| Entrenamiento | 70 % | ~2,692 |
| Validación    | 15 % | ~577   |
| Prueba        | 15 % | ~577   |

Se usa `stratify=etiquetas` para mantener la proporción de clases en cada split.


In [ ]:
# Primera división: 70% train — 30% temp
rutas_train, rutas_temp, etiq_train, etiq_temp = train_test_split(
    rutas, etiquetas,
    test_size=0.30,
    random_state=SEMILLA,
    stratify=etiquetas
)

# Segunda división: 50% val — 50% test del 30% temp → 15% cada uno
rutas_val, rutas_test, etiq_val, etiq_test = train_test_split(
    rutas_temp, etiq_temp,
    test_size=0.50,
    random_state=SEMILLA,
    stratify=etiq_temp
)

total = len(rutas)
print("División del dataset:")
print(f"  Entrenamiento : {len(rutas_train):5d} imágenes  ({len(rutas_train)/total*100:.1f}%)")
print(f"  Validación    : {len(rutas_val):5d} imágenes  ({len(rutas_val)/total*100:.1f}%)")
print(f"  Prueba        : {len(rutas_test):5d} imágenes  ({len(rutas_test)/total*100:.1f}%)")
print(f"  {'TOTAL':14s}: {total:5d} imágenes")


### 2.4 Funciones de carga y preprocesamiento

**Decisiones de preprocesamiento documentadas:**
- Redimensión a **64×64 px** — balance entre detalle visual y velocidad de entrenamiento en GPU
- Normalización a **[0, 1]** dividiendo entre 255 — estabiliza el gradiente descendente
- Canal **RGB** (3 canales) — se descartan imágenes en escala de grises convirtiéndolas a RGB
- Aumentación **solo en entrenamiento**: flip horizontal, brillo y contraste aleatorio — reduce sobreajuste


In [ ]:
# ── Parámetros de imagen ─────────────────────────────────────────────────
ALTO   = IMG_ALTO    # 64
ANCHO  = IMG_ANCHO   # 64

def cargar_imagen(ruta):
    """Lee una imagen desde disco, la convierte a RGB y la redimensiona."""
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)      # fuerza 3 canales RGB
    img = tf.image.resize(img, [ALTO, ANCHO])        # redimensionar a 64×64
    img = tf.cast(img, tf.float32) / 255.0           # normalizar a [0, 1]
    return img

def aumentar(imagen):
    """Aumentación de datos aplicada SOLO al conjunto de entrenamiento."""
    imagen = tf.image.random_flip_left_right(imagen)           # flip horizontal
    imagen = tf.image.random_brightness(imagen, max_delta=0.15) # brillo aleatorio
    imagen = tf.image.random_contrast(imagen, lower=0.85, upper=1.15) # contraste
    imagen = tf.clip_by_value(imagen, 0.0, 1.0)               # mantener rango [0,1]
    return imagen

def procesar_train(ruta, etiqueta):
    """Pipeline de entrenamiento: carga + aumentación."""
    img = cargar_imagen(ruta)
    img = aumentar(img)
    return img, etiqueta

def procesar_eval(ruta, etiqueta):
    """Pipeline de validación/prueba: solo carga, sin aumentación."""
    img = cargar_imagen(ruta)
    return img, etiqueta

print("Funciones de preprocesamiento definidas:")
print("  cargar_imagen()  → lee, redimensiona 64×64, normaliza [0,1]")
print("  aumentar()       → flip, brillo±15%, contraste 85-115%")
print("  procesar_train() → carga + aumentación  (solo entrenamiento)")
print("  procesar_eval()  → solo carga           (validación y prueba)")


### 2.5 Construcción de pipelines `tf.data`

`tf.data` permite cargar imágenes de forma **paralela y prefetcheada**, 
evitando que la CPU sea el cuello de botella mientras la GPU procesa el lote anterior.


In [ ]:
BATCH_SIZE  = 32          # tamaño de lote base (se variará en experimentos)
AUTOTUNE    = tf.data.AUTOTUNE

def construir_pipeline(rutas, etiquetas, batch_size,
                        entrenamiento=False, mezclar=True):
    """
    Construye un pipeline tf.data optimizado para GPU.

    Parámetros
    ----------
    rutas        : array de rutas de imágenes
    etiquetas    : array de etiquetas numéricas
    batch_size   : tamaño del lote
    entrenamiento: si True aplica aumentación de datos
    mezclar      : si True mezcla el dataset en cada época
    """
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    if mezclar:
        ds = ds.shuffle(buffer_size=len(rutas), seed=SEMILLA,
                        reshuffle_each_iteration=True)

    # Aplicar preprocesamiento en paralelo
    fn = procesar_train if entrenamiento else procesar_eval
    ds = ds.map(fn, num_parallel_calls=AUTOTUNE)

    # Agrupar en lotes y prefetch para la GPU
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)

    return ds


# Construir los tres pipelines
ds_train = construir_pipeline(rutas_train, etiq_train,
                               BATCH_SIZE, entrenamiento=True)
ds_val   = construir_pipeline(rutas_val,   etiq_val,
                               BATCH_SIZE, entrenamiento=False)
ds_test  = construir_pipeline(rutas_test,  etiq_test,
                               BATCH_SIZE, entrenamiento=False)

# Información de los pipelines
n_train = len(rutas_train)
n_val   = len(rutas_val)
n_test  = len(rutas_test)

print("Pipelines tf.data construidos:")
print(f"  ds_train : {n_train} imágenes → {n_train // BATCH_SIZE + 1} lotes de {BATCH_SIZE}")
print(f"  ds_val   : {n_val}  imágenes → {n_val  // BATCH_SIZE + 1} lotes de {BATCH_SIZE}")
print(f"  ds_test  : {n_test}  imágenes → {n_test  // BATCH_SIZE + 1} lotes de {BATCH_SIZE}")
print()
print("Configuración:")
print(f"  batch_size     : {BATCH_SIZE}")
print(f"  prefetch       : AUTOTUNE")
print(f"  num_parallel   : AUTOTUNE")
print(f"  shuffle train  : True (reshuffle cada época)")


### 2.6 Verificación de un lote

In [ ]:
# Extraer un lote de entrenamiento y verificar dimensiones y valores
for imagenes_lote, etiquetas_lote in ds_train.take(1):
    imgs = imagenes_lote.numpy()
    etiq = etiquetas_lote.numpy()

print("Verificación del primer lote de entrenamiento:")
print(f"  Forma del lote    : {imgs.shape}")
print(f"  Tipo de dato      : {imgs.dtype}")
print(f"  Valor mínimo      : {imgs.min():.4f}  (esperado ≥ 0.0)")
print(f"  Valor máximo      : {imgs.max():.4f}  (esperado ≤ 1.0)")
print(f"  Valor medio       : {imgs.mean():.4f}")
print(f"  Etiquetas en lote : {sorted(set(etiq.tolist()))}")
print()
print(f"  ✓ Shape correcto  : {imgs.shape == (BATCH_SIZE, ALTO, ANCHO, 3)}")
print(f"  ✓ Rango correcto  : {0.0 <= imgs.min() and imgs.max() <= 1.0}")


### 2.7 Visualización de imágenes preprocesadas y aumentadas

In [ ]:
fig, ejes = plt.subplots(3, 8, figsize=(18, 7))
ejes = ejes.flatten()

for i in range(24):
    ejes[i].imshow(imgs[i])
    ejes[i].set_title(IDX_A_CLASE[etiq[i]].replace("_", "\n"),
                      fontsize=6.5)
    ejes[i].axis("off")

fig.suptitle(
    f"Primer lote de entrenamiento — {BATCH_SIZE} imágenes aumentadas (64×64 px)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()


### 2.8 Distribución de clases por conjunto

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
nombres_clases = [IDX_A_CLASE[i].replace("_", "\n") for i in range(NUM_CLASES)]

for ax, (etiq_set, titulo, color) in zip(ejes, [
    (etiq_train, f"Entrenamiento ({len(etiq_train)})", "#2563EB"),
    (etiq_val,   f"Validación ({len(etiq_val)})",      "#16A34A"),
    (etiq_test,  f"Prueba ({len(etiq_test)})",          "#D97706"),
]):
    conteos = [np.sum(etiq_set == i) for i in range(NUM_CLASES)]
    barras  = ax.bar(range(NUM_CLASES), conteos, color=color, alpha=0.8, edgecolor="white")
    ax.set_xticks(range(NUM_CLASES))
    ax.set_xticklabels(nombres_clases, fontsize=6.5, rotation=45, ha="right")
    ax.set_ylabel("Imágenes")
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.set_facecolor("#F8FAFC")
    for barra, cnt in zip(barras, conteos):
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 1,
                str(cnt), ha="center", va="bottom", fontsize=6)

fig.suptitle("Distribución de clases por conjunto — estratificada",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 2.9 Resumen de decisiones de preprocesamiento

In [ ]:
print("=" * 55)
print("  Resumen — Preprocesamiento del Dataset")
print("=" * 55)
print(f"  Resolución de entrada  : {ALTO}×{ANCHO} px  (3 canales RGB)")
print(f"  Normalización          : píxeles / 255  → [0.0, 1.0]")
print(f"  Aumentación (train)    : flip H · brillo±15% · contraste±15%")
print(f"  Split                  : 70% / 15% / 15%  (estratificado)")
print(f"  Tamaño de lote base    : {BATCH_SIZE}")
print(f"  Pipeline               : tf.data  (paralelo + prefetch AUTOTUNE)")
print()
print(f"  Conjunto entrenamiento : {len(rutas_train):5d} imágenes")
print(f"  Conjunto validación    : {len(rutas_val):5d} imágenes")
print(f"  Conjunto prueba        : {len(rutas_test):5d} imágenes")
print(f"  Total                  : {len(rutas):5d} imágenes")
print("=" * 55)
print()
print("  Paso 2 completado — listos para construir el modelo.")
